# HR 1 — Employee Retention Support

Final Assignment — Data Science for Business

Methodology: CRISP-DM

Business context: Employee retention support.

Objective: Investigate whether employee data can help HR identify employees who may leave the company within the next six months.

Intended use: Support voluntary, human-led retention reviews. The model must not make automatic employment decisions.

This notebook documents the six CRISP-DM phases, from understanding the business problem to evaluating the model and proposing a responsible deployment approach.

# Phase 1 — Business Understanding

## 1.1 Business Problem

The business problem is to identify employees who may leave the company within the next six months. The purpose is to investigate whether employee data can support HR in identifying employees who may benefit from additional retention support.

Employee attrition may create challenges for workforce planning and continuity. However, the dataset does not provide the actual financial cost of employee turnover, so these costs cannot be quantified in this analysis.

## 1.2 Stakeholder

The main stakeholder is the HR team or HR managers. They would review the model’s predictions and decide whether a voluntary retention conversation may be appropriate.

## 1.3 Business Decision

The model is intended to support HR in prioritising employees for a voluntary, human-led retention review.

The model must not make automatic employment decisions. HR should review the available information, consider the individual context, and retain the ability to override a model recommendation.

## 1.4 Prediction Moment

The prediction moment is the employee survey date, while the employee is still working for the company and before the outcome for the following six months is known.

Only information available at this moment should be used as a predictor. Information recorded after an employee leaves must be excluded because it would not have been available when the prediction was made.

## 1.5 Scope

The analysis focuses on predicting attrition_next_6m using information available at the prediction moment.

The project will compare a baseline with a candidate model, evaluate prediction errors, and examine whether results differ across departments.

The project does not aim to establish the causes of employee attrition or prove that the model can prevent employees from leaving.

## 1.6 Success Criteria

The candidate model will be compared with a majority-class baseline. Accuracy alone will not be considered sufficient because the target may be imbalanced.

Precision and recall for employees who leave within the next six months will be examined alongside false positives and false negatives. Results will also be reviewed across departments.

The practical usefulness of the model depends on whether its predictions can support HR decisions within the available review capacity.

The case does not provide confirmed HR review capacity, the relative costs of prediction errors, or an agreed minimum performance threshold. These remain open business questions and must not be treated as known company facts.

## 1.7 Initial Assumptions and Limitations

The analysis assumes that the survey date represents the intended prediction moment and that the target records whether an employee leaves within the following six months.

The target does not necessarily establish whether a departure is voluntary. Therefore, the model may predict attrition without identifying which employees would respond to retention support.

Any real-world use would require further validation, human oversight, and consideration of privacy and fairness.

# Phase 2 — Data Understanding

The purpose of this phase is to understand the dataset before preparing the data or building a predictive model.

We will examine the dataset structure, observation unit, business key, target distribution, missing values, duplicate records, variable distributions, and potential data-quality issues.

The findings from this phase will guide the preparation decisions in Phase 3. Following the instructor’s feedback, each important preparation decision must be supported by evidence, verified after the change, and interpreted in relation to the business problem.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("group_08_hr_1.csv")

df = pd.read_csv(DATA)

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (1902, 12)


,employee_id,department,age,tenure_years,weekly_hours,overtime_flag,job_satisfaction_1_5,monthly_salary_eur,manager_changes_2y,commute_km,exit_interview_score_after_leaving,attrition_next_6m
0,E1000000,Operations,62,13.3,40.0,YES,2,4465.14,2,29.1,NaN,0
1,E1000001,Operations,59,1.1,29.5,Yes,3,3647.38,1,21.1,NaN,0
2,E1000002,sales,54,6.4,43.1,No,1,1538.01,6,11.3,NaN,0
3,E1000003,Sales,31,14.9,31.2,No,5,3499.79,1,6.3,NaN,0
4,E1000004,Operations,36,1.1,43.5,No,4,2859.06,0,37.8,NaN,0


In [2]:
# Check missing values
print("Missing values per column:")
display(df.isna().sum().sort_values(ascending=False))

# Check exact duplicate rows
print("\nExact duplicate rows:", df.duplicated().sum())

# Check employee ID uniqueness
print("Unique employee IDs:", df["employee_id"].nunique())
print("Rows with duplicated employee IDs:",
      df["employee_id"].duplicated(keep=False).sum())

# Check target distribution
print("\nTarget distribution:")
display(df["attrition_next_6m"].value_counts(dropna=False).sort_index())

Missing values per column:


exit_interview_score_after_leaving    1758
monthly_salary_eur                      15
employee_id                              0
department                               0
tenure_years                             0
age                                      0
overtime_flag                            0
weekly_hours                             0
job_satisfaction_1_5                     0
manager_changes_2y                       0
commute_km                               0
attrition_next_6m                        0
dtype: int64


Exact duplicate rows: 2
Unique employee IDs: 1900
Rows with duplicated employee IDs: 4

Target distribution:


attrition_next_6m
0    1758
1     144
Name: count, dtype: int64

## 2.1 Dataset Structure

The dataset contains 1,902 rows and 12 columns. Each row is intended to represent an employee at the survey date. However, the repeated employee identifiers identified below mean that the observation unit and key uniqueness require further investigation.

The employee_id column is the proposed business key, and attrition_next_6m is the target variable.

## 2.2 Missing Values

The exit_interview_score_after_leaving variable contains 1,758 missing values, while monthly_salary_eur contains 15 missing values. All other columns have no missing values.

The exit interview score is recorded after an employee leaves. It is therefore unavailable at the intended prediction moment and represents a potential source of data leakage. It should be excluded from model predictors in Phase 3.

The 15 missing salary values require further investigation before a preparation decision is made.

## 2.3 Duplicate Records and Business Key

The dataset contains 2 exact duplicate rows. It also contains 1,900 unique employee identifiers across 1,902 rows, with 4 rows belonging to duplicated employee identifiers.

These findings suggest that duplicate records or repeated employee observations may be present. Before removing any records, we need to inspect the affected rows and determine whether they are exact duplicates or distinct observations for the same employee.

## 2.4 Target Distribution

The target distribution is:

* attrition_next_6m = 0: 1,758 observations (approximately 92.43%).
* attrition_next_6m = 1: 144 observations (approximately 7.57%).

The target is imbalanced because the positive class represents only a small proportion of the dataset. A majority-class baseline would achieve approximately 92.43% accuracy by predicting class 0 for every observation, but it would fail to identify employees in class 1.

Therefore, accuracy alone is not an adequate measure of model usefulness. Precision, recall, and the types of prediction errors must also be considered during modelling and evaluation.

In [4]:
# Show all rows with duplicated employee IDs
duplicate_ids = df[df["employee_id"].duplicated(keep=False)]

display(
    duplicate_ids.sort_values("employee_id")
)

,employee_id,department,age,tenure_years,weekly_hours,overtime_flag,job_satisfaction_1_5,monthly_salary_eur,manager_changes_2y,commute_km,exit_interview_score_after_leaving,attrition_next_6m
1800,E1001800,Operations,38,4.5,24.9,No,1,4499.04,0,39.3,NaN,0
1900,E1001800,Operations,38,4.5,24.9,No,1,4499.04,0,39.3,NaN,0
1817,E1001817,IT,19,7.9,37.9,Yes,1,3451.11,0,19.7,NaN,0
1901,E1001817,IT,19,7.9,37.9,Yes,1,3451.11,0,19.7,NaN,0


In [5]:
# Inspect exact duplicate rows
exact_duplicates = df[df.duplicated(keep=False)]

print("Rows involved in exact duplicates:", len(exact_duplicates))

display(
    exact_duplicates.sort_values("employee_id")
)

Rows involved in exact duplicates: 4


,employee_id,department,age,tenure_years,weekly_hours,overtime_flag,job_satisfaction_1_5,monthly_salary_eur,manager_changes_2y,commute_km,exit_interview_score_after_leaving,attrition_next_6m
1800,E1001800,Operations,38,4.5,24.9,No,1,4499.04,0,39.3,NaN,0
1900,E1001800,Operations,38,4.5,24.9,No,1,4499.04,0,39.3,NaN,0
1817,E1001817,IT,19,7.9,37.9,Yes,1,3451.11,0,19.7,NaN,0
1901,E1001817,IT,19,7.9,37.9,Yes,1,3451.11,0,19.7,NaN,0


In [6]:
# Inspect rows with missing monthly salary
missing_salary = df[df["monthly_salary_eur"].isna()]

print("Rows with missing salary:", len(missing_salary))

display(missing_salary)

Rows with missing salary: 15


,employee_id,department,age,tenure_years,weekly_hours,overtime_flag,job_satisfaction_1_5,monthly_salary_eur,manager_changes_2y,commute_km,exit_interview_score_after_leaving,attrition_next_6m
138,E1000138,Operations,52,7.5,32.2,No,2,NaN,1,32.8,NaN,0
140,E1000140,Sales,28,6.9,49.0,No,2,NaN,1,4.1,NaN,0
197,E1000197,Finance,21,16.5,47.5,No,5,NaN,0,26.3,NaN,0
397,E1000397,sales,55,11.6,50.8,No,2,NaN,1,12.8,5.0,1
422,E1000422,Finance,47,0.0,33.0,No,1,NaN,0,1.9,NaN,0
664,E1000664,HR,53,12.9,52.3,No,5,NaN,1,10.7,NaN,0
750,E1000750,HR,52,7.6,45.5,No,5,NaN,1,50.5,NaN,0
901,E1000901,Operations,43,0.5,25.2,No,5,NaN,0,10.7,NaN,0
978,E1000978,HR,51,1.6,40.0,Yes,4,NaN,1,18.2,NaN,0
1078,E1001078,Operations,31,1.5,47.3,No,4,NaN,2,33.2,NaN,0


### Missing Salary Values

The dataset contains 15 missing values in `monthly_salary_eur`, representing approximately 0.79% of the 1,902 records. These missing values occur across several departments.

Two of the 15 records have `attrition_next_6m = 1`, while the remaining 13 have `attrition_next_6m = 0`. This observation alone does not establish a relationship between missing salary values and employee attrition.

**Evidence → interpretation → next decision:**

We observed 15 missing salary values across multiple departments. Since the missing values affect a small proportion of the dataset, deleting all affected records may be unnecessary. However, the cause of the missing values is unknown. In Phase 3, we will select and justify an appropriate treatment and verify its effect. Any learned imputation value must be estimated using the training data only.

In [7]:
# Inspect all department labels
print("Department categories:")
display(df["department"].value_counts(dropna=False))

# Inspect all overtime labels
print("\nOvertime categories:")
display(df["overtime_flag"].value_counts(dropna=False))

# Inspect job satisfaction values
print("\nJob satisfaction values:")
display(df["job_satisfaction_1_5"].value_counts(dropna=False).sort_index())

Department categories:


department
Operations    555
Sales         448
IT            391
Finance       243
HR            166
sales          99
Name: count, dtype: int64


Overtime categories:


overtime_flag
No     1305
Yes     376
no      133
YES      88
Name: count, dtype: int64


Job satisfaction values:


job_satisfaction_1_5
1    395
2    376
3    397
4    355
5    379
Name: count, dtype: int64

### Categorical Variables and Data Consistency

The department column contains inconsistent category labels. The values `Sales` and `sales` refer to the same department but are recorded with different capitalisation. Combining these categories gives 547 Sales records and results in five distinct departments.

The `overtime_flag` column also contains inconsistent labels. The values `No` and `no` represent the same category, as do `Yes` and `YES`. After standardisation, the dataset contains 1,438 records with `No` and 464 records with `Yes`.

The `job_satisfaction_1_5` variable contains all five expected values, from 1 to 5. No values outside this range were observed in the frequency counts. However, the distribution alone does not show whether job satisfaction is associated with future attrition.

**Evidence → interpretation → next decision:** We observed inconsistent capitalisation in two categorical variables. In Phase 3, we will standardise these labels and verify the resulting categories and counts. This will prevent the same department or overtime status from being treated as different categories during analysis and modelling.

In [8]:
numeric_columns = [
    "age",
    "tenure_years",
    "monthly_salary_eur",
    "weekly_hours",
    "manager_changes_2y",
    "commute_km"
]

display(df[numeric_columns].describe().round(2))

,age,tenure_years,monthly_salary_eur,weekly_hours,manager_changes_2y,commute_km
count,1902.00,1902.00,1887.00,1902.00,1902.00,1902.00
mean,42.15,5.98,3590.85,39.83,0.80,24.45
std,13.18,3.96,1212.40,7.44,0.92,15.10
min,19.00,0.00,1200.00,0.00,0.00,1.00
25%,31.00,3.00,2763.81,34.90,0.00,12.50
50%,42.00,5.90,3533.69,39.80,1.00,24.40
75%,53.00,8.60,4428.62,44.70,1.00,35.28
max,64.00,19.10,9138.15,95.00,7.00,73.80


### Numerical Variable Distributions

The numerical summary describes six variables: age, tenure, monthly salary, weekly working hours, manager changes during the previous two years, and commuting distance.

The average employee age is 42.15 years, and the average tenure is 5.98 years. The average monthly salary is €3,590.85, calculated from 1,887 non-missing records.

The `weekly_hours` variable ranges from 0 to 95 hours. These extreme values require further inspection because they may represent unusual situations or data-quality issues. The summary statistics alone do not establish whether they are incorrect.

The `manager_changes_2y` variable ranges from 0 to 7, while `commute_km` ranges from 1 to 73.8 kilometres. These ranges describe variation in the dataset but do not establish a relationship with employee attrition.

**Evidence → interpretation → next decision:** The summary reveals potentially unusual values in `weekly_hours` and 15 missing salary values. In Phase 3, we will inspect the relevant records before deciding how to handle them. We will not remove observations solely because their values are extreme.

In [11]:
# Inspect records with unusual weekly working hours
unusual_hours = df[
    (df["weekly_hours"] == 0) |
    (df["weekly_hours"] > 60)
]

print("Records with 0 or more than 60 weekly hours:", len(unusual_hours))
display(
    unusual_hours[
        ["employee_id", "department", "weekly_hours", "attrition_next_6m"]
    ].sort_values("weekly_hours")
)

Records with 0 or more than 60 weekly hours: 10


,employee_id,department,weekly_hours,attrition_next_6m
260,E1000260,HR,0.0,0
1261,E1001261,Sales,0.0,0
1295,E1001295,sales,0.0,0
868,E1000868,IT,61.1,0
777,E1000777,IT,61.3,0
270,E1000270,Operations,61.5,0
10,E1000010,Sales,64.2,0
1346,E1001346,sales,65.2,1
708,E1000708,HR,95.0,0
470,E1000470,Sales,95.0,0


### Inspection of Unusual Weekly Working Hours

The dataset contains 10 records with either zero weekly working hours or more than 60 hours per week. Three employees have zero recorded hours, five have between 61.1 and 65.2 hours, and two have 95 hours.

These values require attention, but they cannot automatically be classified as errors. The dataset does not explain whether zero hours represent a special employment situation or missing information, nor does it confirm whether the highest values are incorrect.

The target values show that nine of these ten records have `attrition_next_6m = 0`, while one has `attrition_next_6m = 1`. This small number of observations is not sufficient to establish a relationship between unusual working hours and future attrition.

**Evidence → interpretation → next decision:** We identified 10 unusual records, including two with 95 weekly hours. Because the meaning of these values is uncertain, we will not remove them solely because they are extreme. We will retain them unless further evidence justifies a different treatment and document this limitation in the evaluation.

In [12]:
print("Columns in the dataset:")
for column in df.columns:
    print("-", column)

print("\nNon-missing values in exit interview score:")
print(df["exit_interview_score_after_leaving"].notna().sum())

print("\nTarget distribution for records with an exit interview score:")
display(
    df.loc[
        df["exit_interview_score_after_leaving"].notna(),
        "attrition_next_6m"
    ].value_counts(dropna=False)
)

Columns in the dataset:
- employee_id
- department
- age
- tenure_years
- weekly_hours
- overtime_flag
- job_satisfaction_1_5
- monthly_salary_eur
- manager_changes_2y
- commute_km
- exit_interview_score_after_leaving
- attrition_next_6m

Non-missing values in exit interview score:
144

Target distribution for records with an exit interview score:


attrition_next_6m
1    144
Name: count, dtype: int64

### Potential Data Leakage

The column `exit_interview_score_after_leaving` contains 144 non-missing values, while 1,758 records have missing values. The column name indicates that the score is collected after an employee leaves.

The prediction is intended to support HR decisions before the outcome is known. Therefore, a score collected after an employee leaves would not be available at the prediction moment.

**Evidence → interpretation → next decision:** We observed that `exit_interview_score_after_leaving` is populated for 144 records and appears to represent post-exit information. This variable will be excluded from the model predictors in Phase 3 to prevent data leakage. We will retain the target `attrition_next_6m` separately as the outcome to predict.

In [13]:
print("Target distribution among records with an exit interview score:")

display(
    df.loc[
        df["exit_interview_score_after_leaving"].notna(),
        "attrition_next_6m"
    ].value_counts(dropna=False).sort_index()
)

Target distribution among records with an exit interview score:


attrition_next_6m
1    144
Name: count, dtype: int64

### Exit Interview Score and Target Distribution

All 144 records with a non-missing `exit_interview_score_after_leaving` value have `attrition_next_6m = 1`. The other 1,758 records have no recorded exit interview score.

This pattern shows that the presence of an exit interview score perfectly coincides with the positive target class in this dataset. However, this association does not establish how the target was created or whether the variable directly determined the target.

The variable is collected after an employee leaves and is therefore unavailable at the intended prediction moment.

**Evidence → interpretation → next decision:** We observed that all 144 records with an exit interview score belong to the positive target class. Because the information is available only after departure, we will exclude this variable from the model predictors to prevent data leakage. The target variable will remain separate and will be used only as the outcome to predict.

In [14]:
# Keep the original dataset unchanged
df_original = df.copy()

# Create a separate working copy for data preparation
df_prepared = df.copy()

print("Original dataset shape:", df_original.shape)
print("Working copy shape:", df_prepared.shape)
print("Original dataset unchanged:", df_original.equals(df))

Original dataset shape: (1902, 12)
Working copy shape: (1902, 12)
Original dataset unchanged: True


### Preparation Decision 1: Removing Exact Duplicate Rows

**Evidence:** During Phase 2, we identified two pairs of exact duplicate rows. Four rows were involved, but each pair contained the same employee ID and identical values across all columns.

**Decision:** We will remove the repeated copies while keeping one copy of each record. This prevents identical records from receiving additional weight during model training and evaluation.

**Verification:** We will compare the dataset dimensions and duplicate counts before and after the operation. The expected result is a reduction from 1,902 to 1,900 rows, with no exact duplicate rows remaining.

**Interpretation and next decision:** If the verification confirms these results, we will proceed to the next preparation decision. Removing these duplicates does not guarantee that every remaining employee record is valid, so other data-quality checks are still necessary.

In [15]:
# Check duplicates before removal
print("Rows before:", len(df_prepared))
print("Exact duplicate rows before:", df_prepared.duplicated().sum())

# Remove exact duplicate rows, keeping the first occurrence
df_prepared = df_prepared.drop_duplicates().copy()

# Verify the result
print("\nRows after:", len(df_prepared))
print("Exact duplicate rows after:", df_prepared.duplicated().sum())
print("Unique employee IDs after:", df_prepared["employee_id"].nunique())
print("Working dataset shape:", df_prepared.shape)
print("Original dataset shape:", df_original.shape)

Rows before: 1902
Exact duplicate rows before: 2

Rows after: 1900
Exact duplicate rows after: 0
Unique employee IDs after: 1900
Working dataset shape: (1900, 12)
Original dataset shape: (1902, 12)
